# Importação das bibliotecas necessárias

In [1]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

# Crie uma instância da classe AlinharETL

In [2]:
# Crie uma instância da classe
bucket = "silver"
datamart = "dynamicscrm"
extrator_bronze = AlinharETL(bucket,datamart)

2024-11-07 18:30:22,074 - INFO - Iniciando Sessão Spark.


# Leitura das tabelas

In [4]:
df_quotedetails = extrator_bronze.criar_view_temporaria('bronze/dynamicscrm/quotedetails','quotedetails')

2024-11-07 18:30:54,619 - INFO - Aguarde... Criando View (bronze/dynamicscrm/quotedetails)
2024-11-07 18:30:54,694 - INFO - View criada com sucesso


In [5]:
df_quotedetails.count()

110455

In [5]:
df_quotes = extrator_bronze.criar_view_temporaria('bronze/dynamicscrm/quotes','quotes')

2024-11-07 18:24:52,945 - INFO - Aguarde... Criando View (bronze/dynamicscrm/quotes)
2024-11-07 18:24:53,163 - INFO - View criada com sucesso


In [6]:
df_products = extrator_bronze.criar_view_temporaria('bronze/dynamicscrm/products','products')

2024-11-07 18:24:53,168 - INFO - Aguarde... Criando View (bronze/dynamicscrm/products)
2024-11-07 18:24:53,360 - INFO - View criada com sucesso


In [7]:
df_accounts = extrator_bronze.criar_view_temporaria('bronze/dynamicscrm/accounts','accounts')

2024-11-07 18:24:53,366 - INFO - Aguarde... Criando View (bronze/dynamicscrm/accounts)
2024-11-07 18:24:53,586 - INFO - View criada com sucesso


# Tratamentos na tabela 

In [8]:
sql_query = """
SELECT
    quotedetails._findes_coligadaid_value_name AS nm_entidade,
    quotedetails.productname AS nm_produto,
    quotedetails._findes_solucaoid_value_name AS dsc_natureza_produto,
    quotedetails._findes_unidadeexecutoraid_value_name AS nm_unidade_operacional,
    CAST(quotedetails.quantity AS INT) AS qt_produto,
    CAST(quotedetails.manualdiscountamount AS DECIMAL(10, 2)) AS vl_total_desconto,
    CAST(quotedetails.extendedamount AS DECIMAL(10, 2)) AS vl_total_servico,
    CAST(quotedetails.baseamount AS DECIMAL(10, 2)) AS vl_unitario_servico,
    quotedetails._quoteid_value AS cod_proposta,
    CAST(quotedetails.quotestatecode AS INT) AS cod_status_proposta,
    stc.statecode_name AS dsc_status_proposta,
    qt.statuscode_name,
    TO_DATE(qt.closedon_name, 'dd/MM/yyyy') AS dt_finalizacao_proposta,
    TO_DATE(SUBSTRING(qt.createdon_name, 1, 10), 'dd/MM/yyyy') AS dt_inicio_proposta,
    p._findes_categoriaid_value_name AS dsc_classe_produto, 
    quotedetails._findes_coligadaid_value AS cod_entidade,
    quotedetails._productid_value AS cod_produto,
    ac.accountid AS cod_cliente,
    ac.findes_cnpj AS nr_documento_cliente,
    ac.findes_razaosocial AS nm_cliente
FROM quotedetails
LEFT JOIN 
    accounts ac ON ac.accountid = quotedetails._findes_clienteservicoid_value
LEFT JOIN 
    products p ON p.productid = quotedetails._productid_value
LEFT JOIN 
    quotes qt ON qt.quoteid = quotedetails._quoteid_value
LEFT JOIN (
    SELECT DISTINCT statecode, statecode_name
    FROM quotes 
) AS stc ON stc.statecode = quotedetails.quotestatecode
"""


In [9]:
df = extrator_bronze.carregar_dados_delta(sql_query)

2024-11-07 18:24:53,600 - INFO - Aguarde... Executando Query (Delta)
2024-11-07 18:24:53,601 - INFO - 
SELECT
    quotedetails._findes_coligadaid_value_name AS nm_entidade,
    quotedetails.productname AS nm_produto,
    quotedetails._findes_solucaoid_value_name AS dsc_natureza_produto,
    quotedetails._findes_unidadeexecutoraid_value_name AS nm_unidade_operacional,
    CAST(quotedetails.quantity AS INT) AS qt_produto,
    CAST(quotedetails.manualdiscountamount AS DECIMAL(10, 2)) AS vl_total_desconto,
    CAST(quotedetails.extendedamount AS DECIMAL(10, 2)) AS vl_total_servico,
    CAST(quotedetails.baseamount AS DECIMAL(10, 2)) AS vl_unitario_servico,
    quotedetails._quoteid_value AS cod_proposta,
    CAST(quotedetails.quotestatecode AS INT) AS cod_status_proposta,
    stc.statecode_name AS dsc_status_proposta,
    qt.statuscode_name,
    TO_DATE(qt.closedon_name, 'dd/MM/yyyy') AS dt_finalizacao_proposta,
    TO_DATE(SUBSTRING(qt.createdon_name, 1, 10), 'dd/MM/yyyy') AS dt_inicio_pr

In [7]:
df_quotedetails.count()

110455

In [10]:
from pyspark.sql import SparkSession

# Crie uma Spark session com configurações para o S3
spark = SparkSession.builder \
    .appName("Exemplo de Gravação em S3") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.endpoint", "minio:9000") \
    .getOrCreate()


In [8]:
# Supondo que df seja seu DataFrame
df_quotedetails.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .csv("s3a://landing/details.csv")


# Gravação no datalake

In [10]:
extrator_bronze.caminho_tabela_delta = 's3a://silver/ServicoDadosIBGE/ServicoDadosIBGECnae'

In [11]:
extrator_bronze.salvar_delta(compliance_flat_ibge_cnae, 'overwrite')

2024-09-25 18:51:07,455 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-25 18:51:26,086 - INFO - Dados persistidos com sucesso
2024-09-25 18:51:26,086 - INFO - s3a://silver/ServicoDadosIBGE/ServicoDadosIBGECnae
2024-09-25 18:51:26,087 - INFO - Aguarde... Realizando optimize
2024-09-25 18:51:28,817 - INFO - Optimize executado com sucesso.
2024-09-25 18:51:28,818 - INFO - Aguarde... Realizando vacuum
2024-09-25 18:51:47,979 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [5]:
extrator_bronze.parar_sessao()

2024-11-02 08:45:47,091 - INFO - Sessão Spark finalizada.
